# Artigo 2 — Predição de Atrasos na Entrega | Olist Dataset

Reprodução fiel da metodologia CRISP-DM descrita no artigo  
**"Predição de Atrasos na Entrega de Pedidos em E-Commerce"**

Pipeline completo: carregamento → EDA → engenharia de atributos → modelagem → avaliação.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

print('Todas as bibliotecas importadas com sucesso.')

Todas as bibliotecas importadas com sucesso.


---
## FASE 1 — Data Understanding

In [2]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
from pathlib import Path

DATA_DIR    = './datasets/'
REPORTS_DIR = Path('./reports')
FIGURES_DIR = REPORTS_DIR / 'figures'
MODELS_DIR  = Path('./models')

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Carregamento dos 6 CSVs ───────────────────────────────────────────────────
orders      = pd.read_csv(DATA_DIR + 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_DIR + 'olist_order_items_dataset.csv')
products    = pd.read_csv(DATA_DIR + 'olist_products_dataset.csv')
sellers     = pd.read_csv(DATA_DIR + 'olist_sellers_dataset.csv')
customers   = pd.read_csv(DATA_DIR + 'olist_customers_dataset.csv')
geolocation = pd.read_csv(DATA_DIR + 'olist_geolocation_dataset.csv')

print(f'orders:      {orders.shape}')
print(f'order_items: {order_items.shape}')
print(f'products:    {products.shape}')
print(f'sellers:     {sellers.shape}')
print(f'customers:   {customers.shape}')
print(f'geolocation: {geolocation.shape}')

orders:      (99441, 8)
order_items: (112650, 7)
products:    (32951, 9)
sellers:     (3095, 4)
customers:   (99441, 5)
geolocation: (1000163, 5)


In [3]:
# ── Filtro: status='delivered' + data de entrega válida ───────────────────────
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

df_filtered = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].notna())
].copy()

print(f'Pedidos após filtro: {df_filtered.shape[0]:,}')
print(f'Pedidos únicos:      {df_filtered["order_id"].nunique():,}')

Pedidos após filtro: 96,470
Pedidos únicos:      96,470


In [4]:
# ── Variável alvo ─────────────────────────────────────────────────────────────
# Comparação direta entre datas (conforme artigo)
df_filtered['target'] = (
    df_filtered['order_delivered_customer_date'] >
    df_filtered['order_estimated_delivery_date']
).astype(int)

n_total   = len(df_filtered)
n_delay   = df_filtered['target'].sum()
pct_delay = n_delay / n_total * 100

print(f'Total de pedidos:    {n_total:,}')
print(f'Pedidos com atraso:  {n_delay:,}')
print(f'Taxa de atraso:      {pct_delay:.2f}%')

# EDA: distribuição usando dt.days (conforme artigo)
df_filtered['diff_days'] = (
    df_filtered['order_delivered_customer_date'] -
    df_filtered['order_estimated_delivery_date']
).dt.days

print(f'Taxa de atraso (dt.days > 0): {(df_filtered["diff_days"] > 0).mean()*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_filtered['target'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black'
)
axes[0].set_title('Distribuição do Target (0=No-delay, 1=Delay)')
axes[0].set_xlabel('Target')
axes[0].set_ylabel('Contagem')
axes[0].set_xticklabels(['No-delay (0)', 'Delay (1)'], rotation=0)

axes[1].hist(df_filtered['diff_days'].clip(-60, 60), bins=60,
             color='steelblue', edgecolor='black', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Prazo estimado')
axes[1].set_title('Distribuição: dias_entrega - prazo_estimado')
axes[1].set_xlabel('Dias (clip +-60)')
axes[1].set_ylabel('Frequência')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_target_distribution.png', dpi=100)
plt.show()
print(f'EDA salvo em {FIGURES_DIR / "eda_target_distribution.png"}')

Total de pedidos:    96,470
Pedidos com atraso:  7,826
Taxa de atraso:      8.11%
Taxa de atraso (dt.days > 0): 6.77%


EDA salvo em reports/figures/eda_target_distribution.png


---
## FASE 2 — Engenharia de Atributos

In [5]:
# ── Junção: order_items + products + sellers ──────────────────────────────────
items_products = order_items.merge(products, on='product_id', how='left')
items_full = items_products.merge(
    sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']],
    on='seller_id', how='left'
)

# volume_cm3 = product_length_cm * product_height_cm * product_width_cm
items_full['volume_cm3'] = (
    items_full['product_length_cm'] *
    items_full['product_height_cm'] *
    items_full['product_width_cm']
)

# Categoria do produto: fillna='outros'
items_full['product_category_name'] = (
    items_full['product_category_name'].fillna('outros')
)

# Agregar por pedido
agg_items = items_full.groupby('order_id').agg(
    price                  = ('price',                   'sum'),
    freight_value          = ('freight_value',           'sum'),
    product_weight_g       = ('product_weight_g',        'sum'),
    volume_cm3             = ('volume_cm3',              'sum'),
    seller_zip_code_prefix = ('seller_zip_code_prefix',  'first'),
    seller_state           = ('seller_state',            'first'),
    product_category_name  = ('product_category_name',   'first'),
).reset_index()

print(f'agg_items shape: {agg_items.shape}')

agg_items shape: (98666, 8)


In [6]:
# ── Junção com customers ──────────────────────────────────────────────────────
df = df_filtered.merge(
    customers[['customer_id', 'customer_zip_code_prefix', 'customer_state']],
    on='customer_id', how='left'
)
df = df.merge(agg_items, on='order_id', how='left')

# Deduplicação por order_id
df = df.drop_duplicates(subset='order_id').reset_index(drop=True)

print(f'df shape após merge e deduplicação: {df.shape}')

df shape após merge e deduplicação: (96470, 19)


In [7]:
# ── Features temporais ────────────────────────────────────────────────────────
df['dia_semana'] = df['order_purchase_timestamp'].dt.dayofweek
df['mes']        = df['order_purchase_timestamp'].dt.month
df['hora']       = df['order_purchase_timestamp'].dt.hour

# mesmo_estado: 1 se seller_state == customer_state
df['mesmo_estado'] = (df['seller_state'] == df['customer_state']).astype(int)

print('Features temporais e mesmo_estado criadas.')
print(df[['dia_semana', 'mes', 'hora', 'mesmo_estado']].describe())

Features temporais e mesmo_estado criadas.
         dia_semana           mes          hora  mesmo_estado
count  96470.000000  96470.000000  96470.000000  96470.000000
mean       2.756494      6.031046     14.773028      0.359708
std        1.967041      3.228479      5.328347      0.479917
min        0.000000      1.000000      0.000000      0.000000
25%        1.000000      3.000000     11.000000      0.000000
50%        3.000000      6.000000     15.000000      0.000000
75%        4.000000      8.000000     19.000000      1.000000
max        6.000000     12.000000     23.000000      1.000000


In [8]:
# ── Distância Haversine (seller -> customer) ──────────────────────────────────

# Agregar geolocalização por prefixo de CEP (média de lat/lng)
geo_agg = geolocation.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean')
).reset_index()

# Junção: coordenadas do vendedor
df = df.merge(
    geo_agg.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
        'lat': 'seller_lat',
        'lng': 'seller_lng'
    }),
    on='seller_zip_code_prefix', how='left'
)

# Junção: coordenadas do cliente
df = df.merge(
    geo_agg.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'lat': 'customer_lat',
        'lng': 'customer_lng'
    }),
    on='customer_zip_code_prefix', how='left'
)

# Implementação vetorizada da fórmula de Haversine
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371.0  # raio da Terra em km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

df['distancia_km'] = haversine_vectorized(
    df['seller_lat'], df['seller_lng'],
    df['customer_lat'], df['customer_lng']
)

print(f'distancia_km - nulos: {df["distancia_km"].isna().sum():,}')
print(f'distancia_km - media: {df["distancia_km"].mean():.1f} km')
print(f'distancia_km - max:   {df["distancia_km"].max():.1f} km')

distancia_km - nulos: 478
distancia_km - media: 600.8 km
distancia_km - max:   8677.9 km


In [9]:
# ── Seleção final das 12 features (Tabela 2 do artigo) ────────────────────────
num_features = [
    'price', 'freight_value', 'product_weight_g', 'volume_cm3',
    'distancia_km', 'dia_semana', 'mes', 'hora', 'mesmo_estado'
]
cat_features = [
    'product_category_name', 'customer_state', 'seller_state'
]

all_features = num_features + cat_features
TARGET = 'target'

# Ordenação temporal (necessária para o split temporal)
df = df.sort_values('order_purchase_timestamp').reset_index(drop=True)

X = df[all_features].copy()
y = df[TARGET].copy()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Total de features: {len(all_features)} (num={len(num_features)}, cat={len(cat_features)})')
print(f'Taxa de atraso (y=1): {y.mean()*100:.2f}%')
print(f'\nFeatures numéricas ({len(num_features)}): {num_features}')
print(f'Features categóricas ({len(cat_features)}): {cat_features}')

X shape: (96470, 12)
y shape: (96470,)
Total de features: 12 (num=9, cat=3)
Taxa de atraso (y=1): 8.11%

Features numéricas (9): ['price', 'freight_value', 'product_weight_g', 'volume_cm3', 'distancia_km', 'dia_semana', 'mes', 'hora', 'mesmo_estado']
Features categóricas (3): ['product_category_name', 'customer_state', 'seller_state']


In [10]:
# ── EDA extra: distribuição da distância e mês por target ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for t, label, color in [(0, 'No-delay', 'steelblue'), (1, 'Delay', 'tomato')]:
    vals = df.loc[df['target'] == t, 'distancia_km'].dropna()
    axes[0].hist(vals.clip(0, 5000), bins=50, alpha=0.6, label=label, color=color)

axes[0].set_title('Distância (km) por Target')
axes[0].set_xlabel('distancia_km (clip 5000 km)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

df.groupby(['mes', 'target']).size().unstack().plot(
    kind='bar', ax=axes[1], color=['steelblue', 'tomato'], edgecolor='black'
)
axes[1].set_title('Pedidos por Mês e Target')
axes[1].set_xlabel('Mês')
axes[1].set_ylabel('Contagem')
axes[1].legend(['No-delay (0)', 'Delay (1)'])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_features.png', dpi=100)
plt.show()
print(f'EDA salvo em {FIGURES_DIR / "eda_features.png"}')

EDA salvo em reports/figures/eda_features.png


### EDA descritiva: taxas de atraso por mês, estado e tipo de pedido

Tabelas reportadas no `RELATORIO_PREDICAO_ATRASOS.md`. Os valores são calculados sobre `df` (96.470 pedidos deduplicados, após junção com customers/sellers e criação de `mesmo_estado`).

In [11]:
# ── EDA: taxa de atraso por mês, estado e tipo (intra/interestadual) ─────────

# Taxa de atraso por mês
print('=' * 60)
print('TAXA DE ATRASO POR MÊS')
print('=' * 60)
taxa_mes = df.groupby('mes')['target'].agg(['mean', 'sum', 'count'])
taxa_mes['mean'] = (taxa_mes['mean'] * 100).round(2)
taxa_mes.columns = ['Taxa_atraso_%', 'Atrasos', 'Total']
taxa_mes_sorted = taxa_mes.sort_values('Taxa_atraso_%', ascending=False)
print(taxa_mes_sorted.to_string())
print(f'\nMês com maior taxa: {taxa_mes_sorted.index[0]} ({taxa_mes_sorted.iloc[0,0]:.2f}%)')
print(f'Mês com menor taxa: {taxa_mes_sorted.index[-1]} ({taxa_mes_sorted.iloc[-1,0]:.2f}%)')

# Taxa de atraso por estado do cliente
print('\n' + '=' * 60)
print('TAXA DE ATRASO POR ESTADO DO CLIENTE (filtro: n >= 100 pedidos)')
print('=' * 60)
taxa_uf = df.groupby('customer_state')['target'].agg(['mean', 'sum', 'count'])
taxa_uf['mean'] = (taxa_uf['mean'] * 100).round(2)
taxa_uf.columns = ['Taxa_atraso_%', 'Atrasos', 'Total']
taxa_uf = taxa_uf[taxa_uf['Total'] >= 100].sort_values('Taxa_atraso_%', ascending=False)
print('Top 5 piores:')
print(taxa_uf.head(5).to_string())
print('\nTop 5 melhores:')
print(taxa_uf.tail(5).to_string())

# Taxa de atraso: interestadual vs intraestadual
print('\n' + '=' * 60)
print('TAXA DE ATRASO: INTERESTADUAL vs INTRAESTADUAL')
print('=' * 60)
taxa_inter = df.groupby('mesmo_estado')['target'].agg(['mean', 'sum', 'count'])
taxa_inter['mean'] = (taxa_inter['mean'] * 100).round(2)
taxa_inter.columns = ['Taxa_atraso_%', 'Atrasos', 'Total']
taxa_inter.index = ['Interestadual (0)', 'Intraestadual (1)']
print(taxa_inter.to_string())

TAXA DE ATRASO POR MÊS
     Taxa_atraso_%  Atrasos  Total
mes                               
3            17.15     1638   9549
11           14.31     1043   7288
2            13.41     1101   8208
12            8.38      462   5514
8             7.58      799  10544
5             6.64      684  10294
1             6.23      487   7819
4             5.96      542   9101
9             5.23      217   4151
10            5.06      240   4743
7             4.08      409  10028
6             2.21      204   9231

Mês com maior taxa: 3 (17.15%)
Mês com menor taxa: 6 (2.21%)

TAXA DE ATRASO POR ESTADO DO CLIENTE (filtro: n >= 100 pedidos)
Top 5 piores:
                Taxa_atraso_%  Atrasos  Total
customer_state                               
AL                      23.93       95    397
MA                      19.67      141    717
PI                      15.97       76    476
CE                      15.32      196   1279
SE                      15.22       51    335

Top 5 melhores:
       

---
## FASE 3 — Modelagem

In [12]:
# ── Split temporal: 80% treino (mais antigos), 20% teste (mais recentes) ──────
split_idx = int(len(df) * 0.80)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print(f'Treino: {X_train.shape[0]:,} amostras  |  Target=1: {y_train.mean()*100:.2f}%')
print(f'Teste:  {X_test.shape[0]:,} amostras  |  Target=1: {y_test.mean()*100:.2f}%')

Treino: 77,176 amostras  |  Target=1: 8.82%
Teste:  19,294 amostras  |  Target=1: 5.29%


In [13]:
# ── Preprocessador ────────────────────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     num_features),
    ('cat', categorical_transformer, cat_features),
])

print('Preprocessador configurado.')

Preprocessador configurado.


In [14]:
# ── Definição dos 3 modelos ───────────────────────────────────────────────────
models = {
    'Regressao Logistica': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        random_state=42
    ),
}

print(f'{len(models)} modelos definidos: {list(models.keys())}')

3 modelos definidos: ['Regressao Logistica', 'Random Forest', 'HistGradientBoosting']


In [15]:
# ── Treinamento com ImbPipeline (SMOTE(0.3) + cada classificador) ─────────────
results = {}
trained_pipes = {}

for name, clf in models.items():
    print(f'\nTreinando: {name} ...')

    pipe = ImbPipeline(steps=[
        ('pre',   preprocessor),
        ('smote', SMOTE(sampling_strategy=0.3, random_state=42)),
        ('clf',   clf),
    ])

    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob)

    results[name] = {
        'Accuracy':  acc,
        'Precision': prec,
        'Recall':    rec,
        'F1':        f1,
        'AUC-ROC':   auc,
    }

    print(f'  Acc={acc:.4f}  Prec={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  AUC-ROC={auc:.4f}')

print('\nTodos os modelos treinados.')


Treinando: Regressao Logistica ...


  Acc=0.6810  Prec=0.0447  Recall=0.2468  F1=0.0757  AUC-ROC=0.4473

Treinando: Random Forest ...


  Acc=0.9453  Prec=0.0952  Recall=0.0039  F1=0.0075  AUC-ROC=0.5321

Treinando: HistGradientBoosting ...


  Acc=0.9470  Prec=0.3333  Recall=0.0010  F1=0.0020  AUC-ROC=0.5739

Todos os modelos treinados.


---
## FASE 4 — Avaliação

In [16]:
# ── Tabela de resultados obtidos ──────────────────────────────────────────────
df_results = pd.DataFrame(results).T.round(4)
df_results.index.name = 'Modelo'

print('=' * 70)
print('RESULTADOS OBTIDOS (neste experimento)')
print('=' * 70)
print(df_results.to_string())
print('=' * 70)

RESULTADOS OBTIDOS (neste experimento)
                      Accuracy  Precision  Recall      F1  AUC-ROC
Modelo                                                            
Regressao Logistica     0.6810     0.0447  0.2468  0.0757   0.4473
Random Forest           0.9453     0.0952  0.0039  0.0075   0.5321
HistGradientBoosting    0.9470     0.3333  0.0010  0.0020   0.5739


In [17]:
# ── Valores de referência do artigo (Tabela 4) ───────────────────────────────
artigo = {
    'Regressao Logistica': {
        'Accuracy': 0.6826, 'Precision': 0.0451,
        'Recall': 0.2478,   'F1': 0.0763, 'AUC-ROC': 0.4465
    },
    'Random Forest': {
        'Accuracy': 0.9456, 'Precision': 0.1500,
        'Recall': 0.0059,   'F1': 0.0113, 'AUC-ROC': 0.5355
    },
    'HistGradientBoosting': {
        'Accuracy': 0.9471, 'Precision': 0.0000,
        'Recall': 0.0000,   'F1': 0.0000, 'AUC-ROC': 0.5932
    },
}

df_artigo = pd.DataFrame(artigo).T.round(4)
df_artigo.index.name = 'Modelo'

print('=' * 70)
print('VALORES DO ARTIGO (Tabela 4)')
print('=' * 70)
print(df_artigo.to_string())
print('=' * 70)

VALORES DO ARTIGO (Tabela 4)
                      Accuracy  Precision  Recall      F1  AUC-ROC
Modelo                                                            
Regressao Logistica     0.6826     0.0451  0.2478  0.0763   0.4465
Random Forest           0.9456     0.1500  0.0059  0.0113   0.5355
HistGradientBoosting    0.9471     0.0000  0.0000  0.0000   0.5932


In [18]:
# ── Comparação lado a lado: Obtido vs Artigo ─────────────────────────────────
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']

rows = []
for model in artigo.keys():
    for metric in metrics_list:
        obtained = results.get(model, {}).get(metric, float('nan'))
        expected = artigo[model][metric]
        diff = obtained - expected
        rows.append({
            'Modelo':    model,
            'Metrica':   metric,
            'Obtido':    round(obtained, 4),
            'Artigo':    expected,
            'Diferenca': round(diff, 4),
        })

df_compare = pd.DataFrame(rows)

print('=' * 80)
print('COMPARACAO: OBTIDO vs ARTIGO (Tabela 4)')
print('=' * 80)
print(df_compare.to_string(index=False))
print('=' * 80)

COMPARACAO: OBTIDO vs ARTIGO (Tabela 4)
              Modelo   Metrica  Obtido  Artigo  Diferenca
 Regressao Logistica  Accuracy  0.6810  0.6826    -0.0016
 Regressao Logistica Precision  0.0447  0.0451    -0.0004
 Regressao Logistica    Recall  0.2468  0.2478    -0.0010
 Regressao Logistica        F1  0.0757  0.0763    -0.0006
 Regressao Logistica   AUC-ROC  0.4473  0.4465     0.0008
       Random Forest  Accuracy  0.9453  0.9456    -0.0003
       Random Forest Precision  0.0952  0.1500    -0.0548
       Random Forest    Recall  0.0039  0.0059    -0.0020
       Random Forest        F1  0.0075  0.0113    -0.0038
       Random Forest   AUC-ROC  0.5321  0.5355    -0.0034
HistGradientBoosting  Accuracy  0.9470  0.9471    -0.0001
HistGradientBoosting Precision  0.3333  0.0000     0.3333
HistGradientBoosting    Recall  0.0010  0.0000     0.0010
HistGradientBoosting        F1  0.0020  0.0000     0.0020
HistGradientBoosting   AUC-ROC  0.5739  0.5932    -0.0193


In [19]:
# ── Visualização comparativa ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bar_width = 0.35
x = np.arange(len(metrics_list))

for ax, model in zip(axes, artigo.keys()):
    obtained_vals = [results.get(model, {}).get(m, 0) for m in metrics_list]
    artigo_vals   = [artigo[model][m] for m in metrics_list]

    bars1 = ax.bar(x - bar_width/2, obtained_vals, bar_width,
                   label='Obtido', color='steelblue', edgecolor='black', alpha=0.85)
    bars2 = ax.bar(x + bar_width/2, artigo_vals,   bar_width,
                   label='Artigo', color='tomato',   edgecolor='black', alpha=0.85)

    ax.set_title(model, fontsize=10, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_list, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Valor')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Comparacao: Metricas Obtidas vs Artigo (Tabela 4)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'comparacao_obtido_vs_artigo.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Grafico salvo em {FIGURES_DIR / "comparacao_obtido_vs_artigo.png"}')

Grafico salvo em reports/figures/comparacao_obtido_vs_artigo.png


In [20]:
# ── Classification report detalhado por modelo ───────────────────────────────
for name, pipe in trained_pipes.items():
    y_pred = pipe.predict(X_test)
    print(f'\n{"="*55}')
    print(f'Classification Report --- {name}')
    print(f'{"="*55}')
    print(classification_report(y_test, y_pred,
                                 target_names=['No-delay', 'Delay'],
                                 zero_division=0))


Classification Report --- Regressao Logistica
              precision    recall  f1-score   support

    No-delay       0.94      0.71      0.81     18273
       Delay       0.04      0.25      0.08      1021

    accuracy                           0.68     19294
   macro avg       0.49      0.48      0.44     19294
weighted avg       0.90      0.68      0.77     19294




Classification Report --- Random Forest
              precision    recall  f1-score   support

    No-delay       0.95      1.00      0.97     18273
       Delay       0.10      0.00      0.01      1021

    accuracy                           0.95     19294
   macro avg       0.52      0.50      0.49     19294
weighted avg       0.90      0.95      0.92     19294




Classification Report --- HistGradientBoosting
              precision    recall  f1-score   support

    No-delay       0.95      1.00      0.97     18273
       Delay       0.33      0.00      0.00      1021

    accuracy                           0.95     19294
   macro avg       0.64      0.50      0.49     19294
weighted avg       0.91      0.95      0.92     19294



---
### Persistência de Artefatos

Salva os resultados (CSVs em `reports/`) e o melhor modelo (`.pkl` em `models/`).
Critério de seleção: maior **AUC-ROC** (capacidade de ranqueamento, independente do threshold).

In [21]:
# ── Persistência de artefatos ─────────────────────────────────────────────────
import joblib

# CSV 1: resultados obtidos neste experimento
df_results.to_csv(REPORTS_DIR / 'comparacao_modelos.csv', index=True)
print(f'Salvo: {REPORTS_DIR / "comparacao_modelos.csv"}')

# CSV 2: comparação obtido vs artigo
df_compare.to_csv(REPORTS_DIR / 'comparacao_obtido_vs_artigo.csv', index=False)
print(f'Salvo: {REPORTS_DIR / "comparacao_obtido_vs_artigo.csv"}')

# Modelo: melhor por AUC-ROC
best_model_name = df_results['AUC-ROC'].idxmax()
best_pipe = trained_pipes[best_model_name]
best_auc = df_results.loc[best_model_name, 'AUC-ROC']

joblib.dump(best_pipe, MODELS_DIR / 'pipeline_atraso.pkl')
print(f'Modelo salvo: {MODELS_DIR / "pipeline_atraso.pkl"}')
print(f'  Criterio: maior AUC-ROC')
print(f'  Selecionado: {best_model_name} (AUC-ROC = {best_auc:.4f})')

Salvo: reports/comparacao_modelos.csv
Salvo: reports/comparacao_obtido_vs_artigo.csv
Modelo salvo: models/pipeline_atraso.pkl
  Criterio: maior AUC-ROC
  Selecionado: HistGradientBoosting (AUC-ROC = 0.5739)


---
## Conclusões

### Reprodução da Metodologia

O pipeline reproduz fielmente a metodologia do artigo:

- **Dados**: 6 CSVs do dataset Olist, filtro `status='delivered'` com data de entrega válida
- **Target**: comparação direta `order_delivered_customer_date > order_estimated_delivery_date`
- **12 features** (Tabela 2): 9 numéricas + 3 categóricas
- **Distância Haversine** vetorizada: média lat/lng por prefixo de CEP
- **Split temporal**: 80%/20% ordenado por `order_purchase_timestamp`
- **ImbPipeline**: SMOTE(sampling_strategy=0.3) → 3 classificadores

### Observações sobre Possíveis Divergências

Diferenças em relação aos valores da Tabela 4 do artigo podem decorrer de:

1. **Versões de bibliotecas**: diferenças em scikit-learn, imbalanced-learn e numpy
2. **Aleatoriedade interna**: mesmo com `random_state=42`, variações de implementação podem ocorrer entre versões
3. **Deduplicação**: a estratégia `first` para vendedor/categoria pode diferir da implementação original
4. **Prefixos de CEP ausentes**: pedidos sem correspondência na geolocalização geram `NaN` em `distancia_km`, tratado pelo `SimpleImputer(strategy='median')` no pipeline
5. **Classe desbalanceada**: com ~6.59% de atrasos, métricas como Precision e F1 são altamente sensíveis ao threshold do classificador
6. **SMOTE**: a reamostragem estocástica pode variar ligeiramente entre versões do imbalanced-learn
